Loading Library

In [1]:
import pandas as pd
import numpy as np
import os

Loading Dataset

In [2]:
# Defining file paths
RAW_DATA_PATH = "../data/raw/mtn_customer_churn.csv"
PROCESSED_DIR = "../data/processed/"
CLEAN_DATA_PATH = os.path.join(PROCESSED_DIR, "mtn_cleaned_customers.csv")

# Loading the Q1 2025 synthetic dataset
if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError(f"Please drop the dataset file at: {RAW_DATA_PATH}")

df = pd.read_csv(RAW_DATA_PATH)
print(f"Dataset successfully loaded! Shape: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()

Dataset successfully loaded! Shape: 974 rows, 17 columns.


,Customer ID,Full Name,Date of Purchase,Age,State,MTN Device,Gender,Satisfaction Rate,Customer Review,Customer Tenure in months,Subscription Plan,Unit Price,Number of Times Purchased,Total Revenue,Data Usage,Customer Churn Status,Reasons for Churn
0,CUST0001,Ngozi Berry,Jan-25,27,Kwara,4G Router,Male,2,Fair,2,165GB Monthly Plan,35000,19,665000,44.48,Yes,Relocation
1,CUST0002,Zainab Baker,Mar-25,16,Abuja (FCT),Mobile SIM Card,Female,2,Fair,22,12.5GB Monthly Plan,5500,12,66000,19.79,Yes,Better Offers from Competitors
2,CUST0003,Saidu Evans,Mar-25,21,Sokoto,5G Broadband Router,Male,1,Poor,60,150GB FUP Monthly Unlimited,20000,8,160000,9.64,No,NaN
3,CUST0003,Saidu Evans,Mar-25,21,Sokoto,Mobile SIM Card,Male,1,Poor,60,1GB+1.5mins Daily Plan,500,8,4000,197.05,No,NaN
4,CUST0003,Saidu Evans,Mar-25,21,Sokoto,Broadband MiFi,Male,1,Poor,60,30GB Monthly Broadband Plan,9000,15,135000,76.34,No,NaN


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Customer ID                974 non-null    object 
 1   Full Name                  974 non-null    object 
 2   Date of Purchase           974 non-null    object 
 3   Age                        974 non-null    int64  
 4   State                      974 non-null    object 
 5   MTN Device                 974 non-null    object 
 6   Gender                     974 non-null    object 
 7   Satisfaction Rate          974 non-null    int64  
 8   Customer Review            974 non-null    object 
 9   Customer Tenure in months  974 non-null    int64  
 10  Subscription Plan          974 non-null    object 
 11  Unit Price                 974 non-null    int64  
 12  Number of Times Purchased  974 non-null    int64  
 13  Total Revenue              974 non-null    int64  

In [4]:
print(df.isnull().sum()[df.isnull().sum() > 0])

Reasons for Churn    690
dtype: int64


Data Cleaning 

handling date column

In [5]:
df['Date of Purchase'].head()

0    Jan-25
1    Mar-25
2    Mar-25
3    Mar-25
4    Mar-25
Name: Date of Purchase, dtype: object

In [6]:
# Converting date format
df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'], format='%b-%y', errors='coerce')

# Check if any rows failed to parse this time
failed_dates = df['Date of Purchase'].isna().sum()
print(f"Date conversion complete. Total unparseable dates: {failed_dates}")

# Peek at the successfully converted formats
print(df['Date of Purchase'].head())

Date conversion complete. Total unparseable dates: 0
0   2025-01-01
1   2025-03-01
2   2025-03-01
3   2025-03-01
4   2025-03-01
Name: Date of Purchase, dtype: datetime64[ns]


Standardizing Categorical Strings & Regional Names.

In [7]:
# Strip spaces and cast categorical features to strings safely
string_cols = ['State', 'MTN Device', 'Gender', 'Customer Review', 'Subscription Plan']
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

# Normalize State formatting to match standard Title Case names
df['State'] = df['State'].str.title()

# Print unique value counts for verification
print("--- Standardized Device Options ---")
print(df['MTN Device'].value_counts())

print("\n--- Unique States Count ---")
print(f"Total unique states recorded: {df['State'].nunique()}")

--- Standardized Device Options ---
MTN Device
Mobile SIM Card        301
5G Broadband Router    229
Broadband MiFi         228
4G Router              216
Name: count, dtype: int64

--- Unique States Count ---
Total unique states recorded: 35


Contextual Null Handling for Churn Reasons

In [8]:
# Check missing values before handling
print(f"Missing values in 'Reasons for Churn' before: {df['Reasons for Churn'].isna().sum()}")

# Fill missing entries with 'Active Customer'
df['Reasons for Churn'] = df['Reasons for Churn'].fillna('Active Customer')

# Verify the fix
print(f"Missing values in 'Reasons for Churn' after: {df['Reasons for Churn'].isna().sum()}")
print("\nSample of unique churn reasons recorded:")
print(df['Reasons for Churn'].unique()[:5])

Missing values in 'Reasons for Churn' before: 690
Missing values in 'Reasons for Churn' after: 0

Sample of unique churn reasons recorded:
['Relocation' 'Better Offers from Competitors' 'Active Customer'
 'Poor Network' 'Costly Data Plans']


In [9]:
calculated_rev = df['Unit Price'] * df['Number of Times Purchased']

# Identifying where the dataset's 'Total Revenue' doesn't match our calculation
mismatches = df[df['Total Revenue'] != calculated_rev]
print(f"Data Integrity Status: Found {len(mismatches)} revenue calculation mismatches.")

# Fix mismatch (If any)
if len(mismatches) > 0:
    print("Correcting revenue mismatches to ensure mathematical integrity...")
    df['Total Revenue'] = calculated_rev
    print("Correction complete!")

Data Integrity Status: Found 0 revenue calculation mismatches.


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Customer ID                974 non-null    object        
 1   Full Name                  974 non-null    object        
 2   Date of Purchase           974 non-null    datetime64[ns]
 3   Age                        974 non-null    int64         
 4   State                      974 non-null    object        
 5   MTN Device                 974 non-null    object        
 6   Gender                     974 non-null    object        
 7   Satisfaction Rate          974 non-null    int64         
 8   Customer Review            974 non-null    object        
 9   Customer Tenure in months  974 non-null    int64         
 10  Subscription Plan          974 non-null    object        
 11  Unit Price                 974 non-null    int64         
 12  Number o

Exporting the Cleaned Dataset.

In [11]:
# Creating the processed directory if it doesn't already exist
os.makedirs("../data/processed/", exist_ok=True)

# Export the cleaned data to a new CSV file
df.to_csv("../data/processed/mtn_cleaned_customers.csv", index=False)
print("Data cleaning notebook complete! 'mtn_cleaned_customers.csv' is safely stored.")

Data cleaning notebook complete! 'mtn_cleaned_customers.csv' is safely stored.
